<a href="https://colab.research.google.com/github/nriceusa/cs501r-hce/blob/main/Lab_Exploration_2_LM_Eval_Harness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab Exploration 2 - LM-Eval-Harness

For this lab, you will explore using the Language Model Evaluation Harness (LM-Eval) from EleutherAI. To start, you may be interested in looking over it's associated paper found [here](https://arxiv.org/pdf/2405.14782). LM-Eval is open source library for running large language models through evaluation.

LM-Eval has a complex infrastructure, and a significant part of this lab is about gaining experience interacting with complicated open-source or research-focused packages. This includes navigation and troubleshooting package issues. This skill will probably be helpful for your final project, and it's also great applied experience.

Navigate to the [lm-eval repository](https://github.com/EleutherAI/lm-evaluation-harness). Look through some of it's documentation and become familiar with what using the package looks like.

Your first task is to get lm-eval setup. You'll notice that the install instructions for lm-eval require you to clone the repository and install with pip. You can do this in Colab Pro by pulling up the terminal: select terminal at the bottom of the page. If you're on the free version, you can also just run the following commands:

In [2]:
%%shell
git clone --depth 1 https://github.com/EleutherAI/lm-evaluation-harness
cd lm-evaluation-harness
pip install -e .

Cloning into 'lm-evaluation-harness'...
remote: Enumerating objects: 16082, done.
remote: Counting objects: 100% (16082/16082), done.
remote: Compressing objects: 100% (6338/6338), done.
remote: Total 16082 (delta 9770), reused 15774 (delta 9714), pack-reused 0 (from 0)
Receiving objects: 100% (16082/16082), 4.44 MiB | 10.98 MiB/s, done.
Resolving deltas: 100% (9770/9770), done.
Obtaining file:///content/lm-evaluation-harness
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 12.0 MB/s eta 0:00:00
 

In [3]:
# Install both hf and vllm backends
!pip install "lm_eval[hf,vllm]" jedi protobuf==5.29.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of vllm to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 12.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of grpcio-reflection to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-reflection to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 862.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 21.0 M

In [1]:
import sys
sys.path.append('/content/lm-evaluation-harness') # Add package directory to path, necessary in Colab
import lm_eval
import json
from lm_eval.utils import handle_non_serializable
from huggingface_hub import login
from lm_eval.utils import setup_logging
# Set log level
setup_logging("DEBUG")  # DEBUG, INFO, WARNING, ERROR

Login to HuggingFace Hub; some datasets and models are gated, meaning you will have to agree to Terms and Conditions before you can use them.

In [2]:
login()

To use LM-Eval, you can directly run it through the shell:

In [3]:
%%shell
lm_eval --model hf \
    --model_args pretrained=openai-community/gpt2 \
    --tasks hellaswag \
    --device cuda:0 \
    --batch_size 16

2026-02-15:02:17:20 INFO     [_cli.run:376] Selected Tasks: ['hellaswag']
2026-02-15:02:17:24 INFO     [evaluator:213] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-15:02:17:24 INFO     [evaluator:238] Initializing hf model, with arguments: {'pretrained': 'openai-community/gpt2'}
2026-02-15:02:17:30 INFO     [models.huggingface:161] Using device 'cuda:0'
config.json: 100% 665/665 [00:00<00:00, 4.86MB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 209kB/s]
vocab.json: 1.04MB [00:00, 11.1MB/s]
merges.txt: 456kB [00:00, 7.96MB/s]
tokenizer.json: 1.36MB [00:00, 10.2MB/s]
2026-02-15:02:17:33 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
2026-02-15 02:17:35.490498: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT whe

Alternatively, you can also use the Python API:

In [5]:
from pprint import pprint

models = ["gpt2"]

all_results = {}
for model_name in models:
    results = lm_eval.simple_evaluate(
        model="hf",
        model_args=f"pretrained={model_name}",
        tasks=["hellaswag"],
        batch_size="auto",
    )
    all_results[model_name] = results["results"]

pprint(all_results)

# Save results
with open("results.json", "w") as f:
    json.dump(results, f, default=handle_non_serializable, indent=2)

2026-02-15:02:30:15 INFO     [evaluator:213] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-15:02:30:15 INFO     [evaluator:251] Initializing hf model, with arguments: {'pretrained': 'gpt2'}
2026-02-15:02:30:15 INFO     [models.huggingface:161] Using device 'cuda'
2026-02-15:02:30:16 DEBUG    [models.huggingface:553] Using model type 'causal'
2026-02-15:02:30:16 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda'}
2026-02-15:02:30:17 DEBUG    [tasks._index:51] Building task index from [PosixPath('/content/lm-evaluation-harness/lm_eval/tasks')]
2026-02-15:02:30:25 DEBUG    [tasks._index:78] Built task index with 14069 entries
2026-02-15:02:30:32 INFO     [evaluator_utils:446] Selected tasks:
2026-02-15:02:30:32 INFO     [evaluator_utils:480] Task: hellaswag (hellaswag/hellaswag.yaml)
2026-02-15:02:30:32 INFO     [api.task:3

Passed argument batch_size = auto:1. Detecting largest batch size
Determined largest batch size: 64


Running loglikelihood requests: 100%|██████████| 40168/40168 [04:39<00:00, 143.58it/s]


{'gpt2': {'hellaswag': {'acc,none': 0.2891854212308305,
                        'acc_norm,none': 0.31139215295757816,
                        'acc_norm_stderr,none': 0.004621163476949437,
                        'acc_stderr,none': 0.004524575892953094,
                        'alias': 'hellaswag',
                        'name': 'hellaswag',
                        'sample_len': 10042}}}


We are expecting you to spend ~5 hours on this assignment exploring and trying to use LM-Eval

For this lab, your tasks are to:
- Read through python api and tasks: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/python-api.md
- Pick at least 3 tasks that sound interesting to you, and learn about each of them.
- Pick 2 models to compare on each of these tasks.
- Attempt to run both models across all tasks, and plot/table results.
- If you run into issues running one of the tasks, dig into the problem a bit and see if you can fix it, or at least understand what is happening (we're expecting most of your time will end up being here).
- Compare the scores to the high scores for each benchmark.

Some tips:
- It might be a good idea to run all of the benchmarks you want to try on a very small model first, to see if they work, before scaling up to a large model.
- Some good small models to test are HuggingFaceTB/SmolLM-360M or meta-llama/Llama-3.2-1B-Instruct. Any larger than 1B and you might start running into memory issues.
- Be aware, some benchmarks are very large and may take a very long time to finish (like, 10+ hours in Colab). For example, bbh. If this is the case, you can try increasing the batch size, or just try a different benchmark.

Other directions you may look into:
- In class, we are reading the MMLU paper. In LM-Eval, you have access to several versions of mmlu, including mmlu_pro, mmlu-pro-plus, mmlu_prox, and mmlusr. Try to run a model across all of these, and compare the differences in scores. What differences do you notice?
- Try using the vllm backend instead of hf, if you haven't yet. It /should/ be faster. (I personally couldn't get it to work)
- Look into what it takes to incorporate a new benchmark into lm-eval.

A final tip: you can track your GPU memory usage by hitting the RAM/Disk usage graphs in the upper right-hand corner. Sometimes, your GPU memory might not be properly released when running these benchmarks. To fix this, you can either restart the kernel (Runtime->Restart session) or try to run the following code:

In [6]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [9]:
# experimentation here

from pprint import pprint

models = [
    "HuggingFaceTB/SmolLM-360M",
    "Qwen/Qwen2.5-1.5B-Instruct"  # Switched to an open model
]

all_results = {}
for model_name in models:
    print(f"Evaluating {model_name}...")
    try:
        results = lm_eval.simple_evaluate(
            model="hf",
            model_args=f"pretrained={model_name}",
            tasks=["mmlu"],
            batch_size=8,
            device="cuda",
        )
        all_results[model_name] = results["results"]
    except Exception as e:
        print(f"Error evaluating {model_name}: {e}")

pprint(all_results)

# Save results
with open("results_mmlu.json", "w") as f:
    json.dump(all_results, f, default=handle_non_serializable, indent=2)

2026-02-15:03:33:20 INFO     [evaluator:213] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-15:03:33:20 INFO     [evaluator:251] Initializing hf model, with arguments: {'pretrained': 'HuggingFaceTB/SmolLM-360M'}
2026-02-15:03:33:20 INFO     [models.huggingface:161] Using device 'cuda'
2026-02-15:03:33:20 DEBUG    [models.huggingface:553] Using model type 'causal'


Evaluating HuggingFaceTB/SmolLM-360M...


2026-02-15:03:33:21 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda'}
2026-02-15:03:33:22 DEBUG    [tasks._index:51] Building task index from [PosixPath('/content/lm-evaluation-harness/lm_eval/tasks')]
2026-02-15:03:33:31 DEBUG    [tasks._index:78] Built task index with 14069 entries
2026-02-15:03:34:22 INFO     [evaluator_utils:446] Selected tasks:
2026-02-15:03:34:22 INFO     [evaluator_utils:462] Group: mmlu
2026-02-15:03:34:22 INFO     [evaluator_utils:462]   Group: mmlu_stem
2026-02-15:03:34:22 INFO     [evaluator_utils:470]     Task: mmlu_abstract_algebra (mmlu/default/mmlu_abstract_algebra.yaml)
2026-02-15:03:34:22 INFO     [evaluator_utils:470]     Task: mmlu_anatomy (mmlu/default/mmlu_anatomy.yaml)
2026-02-15:03:34:22 INFO     [evaluator_utils:470]     Task: mmlu_astronomy (mmlu/default/mmlu_astronomy.yaml)
2026-02-15:03:34:22 INFO     [evaluator_utils:470]     Task: mmlu_college_biology (mmlu/defa

Evaluating Qwen/Qwen2.5-1.5B-Instruct...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

...

2026-02-15:03:46:15 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda'}


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

2026-02-15:03:46:50 DEBUG    [tasks._index:51] Building task index from [PosixPath('/content/lm-evaluation-harness/lm_eval/tasks')]
2026-02-15:03:47:00 DEBUG    [tasks._index:78] Built task index with 14069 entries
2026-02-15:03:47:44 INFO     [evaluator_utils:446] Selected tasks:
2026-02-15:03:47:44 INFO     [evaluator_utils:462] Group: mmlu
2026-02-15:03:47:44 INFO     [evaluator_utils:462]   Group: mmlu_stem
2026-02-15:03:47:44 INFO     [evaluator_utils:470]     Task: mmlu_abstract_algebra (mmlu/default/mmlu_abstract_algebra.yaml)
2026-02-15:03:47:44 INFO     [evaluator_utils:470]     Task: mmlu_anatomy (mmlu/default/mmlu_anatomy.yaml)
2026-02-15:03:47:44 INFO     [evaluator_utils:470]     Task: mmlu_astronomy (mmlu/default/mmlu_astronomy.yaml)
2026-02-15:03:47:44 INFO     [evaluator_utils:470]     Task: mmlu_college_biology (mmlu/default/mmlu_college_biology.yaml)
2026-02-15:03:47:44 INFO     [evaluator_utils:470]     Task: mmlu_college_chemistry (mmlu/default/mmlu_college_chemistr

{'HuggingFaceTB/SmolLM-360M': {'mmlu': {'acc,none': 0.2469733656174334,
                                        'acc_stderr,none': np.float64(0.0036376681899198352),
                                        'alias': 'mmlu',
                                        'name': 'mmlu',
                                        'sample_count': {'acc,none': 14042},
                                        'sample_len': 14042},
                               'mmlu_abstract_algebra': {'acc,none': 0.22,
                                                         'acc_stderr,none': 0.041633319989322654,
                                                         'alias': 'abstract_algebra',
                                                         'name': 'mmlu_abstract_algebra',
                                                         'sample_len': 100},
                               'mmlu_anatomy': {'acc,none': 0.2222222222222222,
                                                'acc_stderr,none': 0.0359144

In [12]:
models = [
    "HuggingFaceTB/SmolLM-360M",
    "Qwen/Qwen2.5-1.5B-Instruct",
]

tasks = [
    "ifeval", # Measures fulfillment of basic, verifiable language instructions
    "musr",   # Measures complex narrative-based reasoning (murder mysteries)
    "moral_stories",  # Measures comprehension of social norms
]

all_results = {}
for model_name in models:
    print(f"Evaluating {model_name} on {tasks}...")
    try:
        results = lm_eval.simple_evaluate(
            model="hf",
            model_args=f"pretrained={model_name}",
            tasks=tasks,
            batch_size=8,
            device="cuda",
            apply_chat_template=True,
        )
        all_results[model_name] = results["results"]
    except Exception as e:
        print(f"Error evaluating {model_name}: {e}")

pprint(all_results)

# Save results
with open("results_other.json", "w") as f:
    json.dump(all_results, f, default=handle_non_serializable, indent=2)


When you are finished, upload to LearningSuite:

1. Your Python notebook
2. A PDF document containing answers to the following questions:

- What did you do? Why did you choose to do this?
- What did you learn from this experimentation?
- Why do you think this is interesting? Or, how can you apply this to the class or your own research?
- Did you spend adequate (~5 hours) of time on this?

Feel free to attach any images outlining your results to your report.